# Anomaly Detection - Finding the Unusual

## 🎯 Learning Objectives

By the end of this notebook, you will understand:
- What are anomalies and why they matter
- Types of anomaly detection algorithms
- Statistical methods (Z-score, IQR)
- Isolation Forest algorithm
- Local Outlier Factor (LOF)
- Robust Covariance (Elliptic Envelope)
- Ensemble anomaly detection

**Estimated Time**: 60-90 minutes

---

## 1. What Are Anomalies?

### Definition
**Anomalies** (also called outliers) are data points that differ significantly from the majority of the data.

### Real-World Examples
- **Credit Card Fraud**: Unusual spending patterns
- **Network Security**: Suspicious traffic patterns
- **Manufacturing**: Defective products
- **Healthcare**: Abnormal patient vitals
- **Energy**: Equipment failures, consumption spikes

### Why Detect Anomalies?
- 🚨 **Prevent problems** before they escalate
- 💰 **Save money** (detect waste, fraud)
- 🔒 **Security** (identify threats)
- ⚙️ **Maintenance** (predictive, not reactive)

### Types of Anomalies

1. **Point Anomalies**: Single unusual data point
   - Example: Energy consumption of 10,000 kWh when normal is 1,500 kWh

2. **Contextual Anomalies**: Unusual in specific context
   - Example: High energy at 3 AM (fine during day)

3. **Collective Anomalies**: Group of points is unusual
   - Example: Gradual increase over several days

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## 2. Statistical Methods for Anomaly Detection

### Method 1: Z-Score (Standard Score)

**How it works**:
- Measures how many standard deviations a point is from the mean
- Formula: `z = (x - μ) / σ`
- Typically, |z| > 3 is considered an anomaly

**Pros**: Simple, fast, interpretable  
**Cons**: Assumes normal distribution, sensitive to extreme outliers

In [ ]:
# Generate sample data with anomalies
np.random.seed(42)

# Normal data
normal_data = np.random.normal(1500, 150, 200)

# Add some anomalies
anomalies = np.array([2800, 2900, 600, 700, 3000])

# Combine
data = np.concatenate([normal_data, anomalies])

print(f"Total data points: {len(data)}")
print(f"Known anomalies: {len(anomalies)}")
print(f"\nData statistics:")
print(f"  Mean: {data.mean():.2f}")
print(f"  Std Dev: {data.std():.2f}")
print(f"  Min: {data.min():.2f}")
print(f"  Max: {data.max():.2f}")

In [ ]:
# Z-Score Anomaly Detection
def detect_anomalies_zscore(data, threshold=3):
    """
    Detect anomalies using Z-score method
    
    Args:
        data: Array of values
        threshold: Number of standard deviations (default: 3)
    
    Returns:
        Boolean array indicating anomalies
    """
    mean = np.mean(data)
    std = np.std(data)
    
    # Calculate z-scores
    z_scores = np.abs((data - mean) / std)
    
    # Flag anomalies
    anomalies = z_scores > threshold
    
    return anomalies, z_scores

# Apply Z-score method
anomaly_flags, z_scores = detect_anomalies_zscore(data, threshold=3)

print(f"Anomalies detected: {anomaly_flags.sum()}")
print(f"\nAnomalous values:")
print(data[anomaly_flags])
print(f"\nTheir Z-scores:")
print(z_scores[anomaly_flags])

In [ ]:
# Visualize Z-score detection
plt.figure(figsize=(14, 6))

# Plot 1: Data distribution
plt.subplot(1, 2, 1)
plt.hist(data[~anomaly_flags], bins=30, alpha=0.7, label='Normal', color='green')
plt.hist(data[anomaly_flags], bins=10, alpha=0.7, label='Anomaly', color='red')
plt.axvline(data.mean(), color='blue', linestyle='--', label='Mean')
plt.axvline(data.mean() + 3*data.std(), color='orange', linestyle=':', label='±3σ')
plt.axvline(data.mean() - 3*data.std(), color='orange', linestyle=':')
plt.xlabel('Energy (kWh)')
plt.ylabel('Frequency')
plt.title('Z-Score Anomaly Detection')
plt.legend()
plt.grid(alpha=0.3)

# Plot 2: Z-scores
plt.subplot(1, 2, 2)
colors = ['red' if a else 'green' for a in anomaly_flags]
plt.scatter(range(len(data)), z_scores, c=colors, alpha=0.6)
plt.axhline(y=3, color='orange', linestyle='--', label='Threshold (3σ)')
plt.xlabel('Data Point Index')
plt.ylabel('|Z-Score|')
plt.title('Z-Scores of All Points')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Method 2: IQR (Interquartile Range)

**How it works**:
- Based on quartiles (25th and 75th percentiles)
- IQR = Q3 - Q1
- Outliers: < Q1 - 1.5×IQR or > Q3 + 1.5×IQR

**Pros**: Robust to extreme values, doesn't assume distribution  
**Cons**: Less sensitive than Z-score

In [ ]:
# IQR Anomaly Detection
def detect_anomalies_iqr(data):
    """
    Detect anomalies using IQR method
    """
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    anomalies = (data < lower_bound) | (data > upper_bound)
    
    return anomalies, lower_bound, upper_bound

# Apply IQR method
iqr_anomalies, lower, upper = detect_anomalies_iqr(data)

print(f"IQR Method Results:")
print(f"  Q1 (25th percentile): {np.percentile(data, 25):.2f}")
print(f"  Q3 (75th percentile): {np.percentile(data, 75):.2f}")
print(f"  IQR: {np.percentile(data, 75) - np.percentile(data, 25):.2f}")
print(f"  Lower bound: {lower:.2f}")
print(f"  Upper bound: {upper:.2f}")
print(f"\nAnomalies detected: {iqr_anomalies.sum()}")
print(f"Anomalous values: {data[iqr_anomalies]}")

In [ ]:
# Compare Z-score vs IQR
comparison = pd.DataFrame({
    'Method': ['Z-Score (3σ)', 'IQR'],
    'Anomalies Detected': [anomaly_flags.sum(), iqr_anomalies.sum()],
    'True Positives': [np.sum(data[anomaly_flags] >= 2500), 
                       np.sum(data[iqr_anomalies] >= 2500)]
})

print("\nComparison:")
print(comparison.to_string(index=False))

# Visualize both methods
plt.figure(figsize=(10, 6))
plt.boxplot(data, vert=True)
plt.scatter(np.ones(anomaly_flags.sum()), data[anomaly_flags], 
           color='red', s=100, label='Z-Score Anomalies', zorder=5, marker='x')
plt.scatter(np.ones(iqr_anomalies.sum())*1.05, data[iqr_anomalies], 
           color='orange', s=80, label='IQR Anomalies', zorder=5, marker='o', alpha=0.6)
plt.ylabel('Energy (kWh)')
plt.title('Box Plot with Anomalies Highlighted')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 3. Machine Learning Methods

### Isolation Forest

**How it works**:
1. Randomly selects a feature
2. Randomly selects a split value
3. Builds a tree by repeating steps 1-2
4. Anomalies are **easier to isolate** (fewer splits needed)

**Key Insight**: Outliers can be isolated quickly with fewer splits!

**Pros**: Fast, works in high dimensions, no assumptions about data distribution  
**Cons**: Requires tuning contamination parameter

In [ ]:
from sklearn.ensemble import IsolationForest

# Prepare data (needs to be 2D for sklearn)
X = data.reshape(-1, 1)

# Train Isolation Forest
iso_forest = IsolationForest(
    contamination=0.05,  # Expected % of anomalies
    random_state=42
)

# Fit and predict (-1 = anomaly, 1 = normal)
predictions = iso_forest.fit_predict(X)
iso_anomalies = predictions == -1

# Get anomaly scores (lower = more anomalous)
scores = iso_forest.score_samples(X)

print(f"Isolation Forest Results:")
print(f"  Anomalies detected: {iso_anomalies.sum()}")
print(f"  Anomalous values: {data[iso_anomalies]}")
print(f"\n  Most anomalous (lowest score): {data[np.argmin(scores)]}")
print(f"  Least anomalous (highest score): {data[np.argmax(scores)]}")

In [ ]:
# Visualize Isolation Forest results
plt.figure(figsize=(14, 6))

# Plot 1: Anomaly scores
plt.subplot(1, 2, 1)
colors = ['red' if a else 'green' for a in iso_anomalies]
plt.scatter(range(len(data)), scores, c=colors, alpha=0.6)
plt.xlabel('Data Point Index')
plt.ylabel('Anomaly Score (lower = more anomalous)')
plt.title('Isolation Forest Anomaly Scores')
plt.grid(alpha=0.3)

# Plot 2: Distribution
plt.subplot(1, 2, 2)
plt.hist(data[~iso_anomalies], bins=30, alpha=0.7, label='Normal', color='green')
plt.hist(data[iso_anomalies], bins=10, alpha=0.7, label='Anomaly', color='red')
plt.xlabel('Energy (kWh)')
plt.ylabel('Frequency')
plt.title('Isolation Forest Detection Results')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Local Outlier Factor (LOF)

**How it works**:
1. Finds k nearest neighbors for each point
2. Calculates local density
3. Compares each point's density to neighbors
4. Points in low-density regions = anomalies

**Key Insight**: Anomalies have lower density than their neighbors!

**Pros**: Detects local outliers, handles varying density  
**Cons**: Sensitive to k parameter, computationally expensive

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

# Train LOF
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)

# Fit and predict
lof_predictions = lof.fit_predict(X)
lof_anomalies = lof_predictions == -1

# Get negative outlier factors (more negative = more anomalous)
lof_scores = lof.negative_outlier_factor_

print(f"LOF Results:")
print(f"  Anomalies detected: {lof_anomalies.sum()}")
print(f"  Anomalous values: {data[lof_anomalies]}")
print(f"\n  Most anomalous score: {lof_scores.min():.3f}")
print(f"  Least anomalous score: {lof_scores.max():.3f}")

### Robust Covariance (Elliptic Envelope)

**How it works**:
1. Assumes data follows Gaussian (normal) distribution
2. Fits an ellipse around the data
3. Points outside the ellipse = anomalies

**Pros**: Fast, works well for Gaussian data  
**Cons**: Assumes normal distribution

In [ ]:
from sklearn.covariance import EllipticEnvelope

# Train Elliptic Envelope
elliptic = EllipticEnvelope(
    contamination=0.05,
    random_state=42
)

# Fit and predict
elliptic_predictions = elliptic.fit_predict(X)
elliptic_anomalies = elliptic_predictions == -1

print(f"Elliptic Envelope Results:")
print(f"  Anomalies detected: {elliptic_anomalies.sum()}")
print(f"  Anomalous values: {data[elliptic_anomalies]}")

## 4. Ensemble Anomaly Detection

**Idea**: Combine multiple methods for more robust detection!

**Voting Strategy**:
- If 2 out of 3 models agree → anomaly
- Reduces false positives
- More confident predictions

In [ ]:
# Ensemble voting
votes = iso_anomalies.astype(int) + lof_anomalies.astype(int) + elliptic_anomalies.astype(int)
ensemble_anomalies = votes >= 2  # At least 2 models agree

print("Ensemble Voting Results:")
print("=" * 50)
print(f"Isolation Forest:    {iso_anomalies.sum()} anomalies")
print(f"LOF:                 {lof_anomalies.sum()} anomalies")
print(f"Elliptic Envelope:   {elliptic_anomalies.sum()} anomalies")
print(f"\nEnsemble (≥2 votes): {ensemble_anomalies.sum()} anomalies")
print(f"\nEnsemble anomalous values:")
print(data[ensemble_anomalies])

In [ ]:
# Visualize voting
results_df = pd.DataFrame({
    'Value': data,
    'Isolation_Forest': iso_anomalies,
    'LOF': lof_anomalies,
    'Elliptic_Envelope': elliptic_anomalies,
    'Votes': votes,
    'Ensemble_Anomaly': ensemble_anomalies
})

# Show voting breakdown
print("\nVoting Breakdown:")
vote_counts = results_df['Votes'].value_counts().sort_index()
for v, count in vote_counts.items():
    print(f"  {v} vote(s): {count} points ({count/len(data)*100:.1f}%)")

# Visualize
plt.figure(figsize=(14, 6))

# Plot 1: Vote distribution
plt.subplot(1, 2, 1)
colors_map = {0: 'green', 1: 'yellow', 2: 'orange', 3: 'red'}
colors = [colors_map[v] for v in votes]
plt.scatter(range(len(data)), data, c=colors, alpha=0.6, s=50)
plt.xlabel('Data Point Index')
plt.ylabel('Energy (kWh)')
plt.title('Ensemble Voting (Green=0, Yellow=1, Orange=2, Red=3 votes)')
plt.grid(alpha=0.3)

# Plot 2: Model agreement
plt.subplot(1, 2, 2)
plt.bar(vote_counts.index, vote_counts.values, 
        color=['green', 'yellow', 'orange', 'red'])
plt.xlabel('Number of Models Flagging as Anomaly')
plt.ylabel('Count')
plt.title('Model Agreement Distribution')
plt.xticks([0, 1, 2, 3])
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Comparing All Methods

In [ ]:
# Create comprehensive comparison
comparison = pd.DataFrame({
    'Method': ['Z-Score', 'IQR', 'Isolation Forest', 'LOF', 'Elliptic Envelope', 'Ensemble'],
    'Anomalies': [
        anomaly_flags.sum(),
        iqr_anomalies.sum(),
        iso_anomalies.sum(),
        lof_anomalies.sum(),
        elliptic_anomalies.sum(),
        ensemble_anomalies.sum()
    ]
})

comparison['Percentage'] = (comparison['Anomalies'] / len(data) * 100).round(2)

print("\nMethod Comparison:")
print("=" * 60)
print(comparison.to_string(index=False))

# Visualize comparison
plt.figure(figsize=(10, 6))
plt.barh(comparison['Method'], comparison['Anomalies'], color='steelblue')
plt.xlabel('Number of Anomalies Detected')
plt.title('Anomaly Detection Method Comparison')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. When to Use Which Method?

| Method | Best For | Avoid When |
|--------|----------|------------|
| **Z-Score** | Quick baseline, normally distributed data | Heavy-tailed distributions, many outliers |
| **IQR** | Robust detection, any distribution | Need high sensitivity |
| **Isolation Forest** | High dimensions, no distribution assumptions | Small datasets (<100 points) |
| **LOF** | Local outliers, varying density | Large datasets (slow) |
| **Elliptic Envelope** | Gaussian data, speed important | Non-Gaussian data |
| **Ensemble** | Production systems, accuracy critical | Need explainability |

## 7. Best Practices

1. **Start Simple**: Try Z-score or IQR first
2. **Understand Your Data**: Is it normal? Multi-modal?
3. **Tune Parameters**: `contamination` is critical
4. **Use Ensemble**: Combine methods for robustness
5. **Validate Results**: Manually check detected anomalies
6. **Domain Knowledge**: Statistical methods + expert input = best results
7. **Monitor Performance**: Track false positives/negatives

## 🎯 Key Takeaways

You now understand:

✅ **What anomalies are** and why they matter  
✅ **Statistical methods**: Z-score and IQR  
✅ **ML methods**: Isolation Forest, LOF, Robust Covariance  
✅ **Ensemble approach**: Combining models for better results  
✅ **When to use** each method  

## ➡️ Next Steps

You're now ready for the main project!

**Go to**: `notebooks/01_data_exploration.ipynb`

Apply these techniques to real energy data from 1,636 buildings!

---

**Congratulations! You're ready to detect anomalies in real-world data!** 🚀